In [ ]:
import asyncio
import os
import re
import sys
from datetime import datetime, timedelta

# ==========================================
# 1. 런타임 환경 검증 및 필수 의존성 동적 설치
# ==========================================
try:
    import google.cloud.storage
    import pandas as pd
    import playwright
except ImportError:
    print("📦 파이프라인 가동을 위한 필수 라이브러리 설치 중...")
    # 서버리스/코랩 컨테이너 환경을 위한 가상 패키지 설치 및 브라우저 의존성 로드
    !pip install -q playwright pandas google-cloud-storage
    !playwright install chromium
    !playwright install-deps chromium
    print("✨ 필수 라이브러리 및 Chromium 드라이버 설치 완료")

# 라이브러리 최종 임포트
from google.cloud import storage
import pandas as pd
from playwright.async_api import async_playwright

print(f"✅ [{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] GFA Scraper 인제스션 환경 세팅 완료")

In [ ]:
import asyncio
from datetime import datetime, timedelta
import os
import re
from google.cloud import storage
import pandas as pd
from playwright.async_api import async_playwright

# =========================================================================
# 1. 파이프라인 구성 설정 및 동적 파라미터 제어 (Quasi-API URL 생성)
# =========================================================================
BUCKET_NAME = "YOUR_SECURE_BUCKET"
GCS_AUTH_PATH = "gfa_auth.json"
LOCAL_AUTH_PATH = "/tmp/gfa_auth.json"
AD_ACCOUNT_ID = "YOUR_GFA_ACCOUNT_ID"

# 윈도우 기반 배치 수집 기간 설정 (최근 10일치 트렌드 스캔)
today_dt = datetime.now()
start_date = (today_dt - timedelta(days=9)).strftime("%Y-%m-%d")
end_date = today_dt.strftime("%Y-%m-%d")

# UI 동선을 우회하는 파라미터 기반 다이렉트 쿼리 스트링 URL 설계
TARGET_URL = (
    f"https://ads.naver.com/manage/ad-accounts/{AD_ACCOUNT_ID}/da/report/performance"
    f"?adUnit=CREATIVE&page=1&filterOptions=%5B%5D"
    f"&showColList=sales%2Cconv_count%2Cconv_sales_krw"
    f"&dateRange={start_date}%2C{end_date}"
    f"&dateUnit=DAY&placeUnit=TOTAL&audience=TOTAL"
)


# =========================================================================
# 2. 메인 스크래핑 및 자가치유형 데이터 인제스션 코어 엔진
# =========================================================================
async def get_gfa_daily_report() -> pd.DataFrame:
    print("🔐 [Step 1] GCP Cloud Storage에서 최신 세션(쿠키) 인증 정보 동기화 중...")
    try:
        storage_client = storage.Client()
        bucket = storage_client.bucket(BUCKET_NAME)
        blob = bucket.blob(GCS_AUTH_PATH)
        blob.download_to_filename(LOCAL_AUTH_PATH)
    except Exception as e:
        print(f"❌ 인증 파일 동기화 실패: {e}")
        return pd.DataFrame()

    async with async_playwright() as p:
        print("🌐 [Step 2] Headless Chromium 브라우저 인스턴스 가동 및 컨텍스트 주입...")
        browser = await p.chromium.launch(headless=True)

        # 가상 스크롤 렌더링 영역 확장을 위한 고해상도 Viewport 설정 (DOM 단절 방지)
        context = await browser.new_context(
            storage_state=LOCAL_AUTH_PATH,
            viewport={"width": 1920, "height": 2000},
        )
        page = await context.new_page()

        print(f"📅 [Step 3] 타겟 매체 다이렉트 랜딩 및 비동기 테이블 요소 대기...")
        await page.goto(TARGET_URL, wait_until="networkidle")
        await page.wait_for_selector(".ad-cms-table-row", timeout=40000)

        # [테크닉 1] 비동기 데이터 지연 로딩 레이어 선제적 확장 처리
        print("🔓 [Step 4] 가상 스크롤 데이터 바인딩을 위한 하위 분기 노드('열기') 전체 개방...")
        await page.hover(".ad-cms-table-tbody-virtual-holder")
        for _ in range(20):
            await page.evaluate("""
                document.querySelectorAll('button[aria-label="열기"]').forEach(btn => btn.click());
            """)
            await page.mouse.wheel(0, 800)
            await page.wait_for_timeout(500)

        print("🔝 [Step 5] 초정밀 그물망 스캔을 위한 스크롤 최상단 리셋...")
        await page.evaluate(
            "document.querySelector('.ad-cms-table-tbody-virtual-holder').scrollTop = 0;"
        )
        await page.wait_for_timeout(2000)

        # [테크닉 2] 자가 치유형 헤더 매칭 및 계층 구조 족보 유지 브릿지 JS 컴포넌트
        JS_EXTRACT_CODE = r"""
        (states) => {
            // 💡 Self-Healing Indexing: 고정된 하드코딩 인덱스 대신 런타임 시점의 헤더 텍스트 매핑으로 UI 변경 대응
            const getIdx = (key) => Array.from(document.querySelectorAll('.ad-cms-table-thead th'))
                                        .findIndex(th => th.querySelector(`[data-column-key="${key}"]`));

            const idx_creative = getIdx("creative"), idx_group = getIdx("ad_set"), idx_campaign = getIdx("campaign");
            const idx_date = getIdx("date_day"), idx_cost = getIdx("sales"), idx_conv = getIdx("conv_count"), idx_conv_sales = getIdx("conv_sales_krw");

            // 고정 레이어 엘리먼트 내 텍스트 및 고유 식별 ID 파싱 유틸리티
            const extractNameAndId = (cell) => {
                if (!cell) return { name: "", id: "" };
                const nameDiv = cell.querySelector('div[style*="ellipsis"]');
                const idDiv = cell.querySelector('div[style*="153, 153, 153"]');
                if (nameDiv && idDiv) return { name: nameDiv.textContent.trim(), id: idDiv.textContent.trim() };
                const text = cell.innerText || "";
                if (text.includes('\n')) {
                    const pts = text.split('\n');
                    return { name: pts[0].trim(), id: pts[pts.length-1].trim() };
                }
                return { name: text.trim(), id: "" };
            };

            let { cur_c, cur_c_id, cur_g, cur_g_id, cur_cp, cur_cp_id } = states;
            let results = [];

            // 💡 가상 돔 소실에 따른 부모-자식 계층 꼬임 방지를 위한 Y좌표 정렬 알고리즘
            const rows = Array.from(document.querySelectorAll('.ad-cms-table-row:not(.ad-cms-table-row-extra)'));
            rows.sort((a, b) => a.getBoundingClientRect().top - b.getBoundingClientRect().top);

            rows.forEach(row => {
                const cells = row.querySelectorAll('.ad-cms-table-cell');
                if (cells.length <= idx_date) return;
                let date_text = cells[idx_date].textContent.trim();

                if (date_text.includes("전체")) {
                    let rKey = row.getAttribute('data-row-key');
                    // 스크롤 시 이름표가 증발하는 현상을 방지하기 위해 extra 고정 레이어 역추적
                    let extra = document.querySelector(`.ad-cms-table-row-extra[data-row-key="${rKey}"]`);
                    let target = extra ? extra.querySelectorAll('.ad-cms-table-cell') : cells;

                    let c_info = extractNameAndId(target[idx_creative]);
                    if (c_info.name) {
                        cur_c = c_info.name; cur_c_id = c_info.id;
                        cur_g = extractNameAndId(target[idx_group]).name; cur_g_id = extractNameAndId(target[idx_group]).id;
                        cur_cp = extractNameAndId(target[idx_campaign]).name; cur_cp_id = extractNameAndId(target[idx_campaign]).id;
                    }
                } else if (/\d/.test(date_text) && cur_c) {
                    // 특수문자 거르고 완전한 정수형 데이터로 정제 (Regex Sanitization)
                    const clean = (c) => c ? parseInt(c.textContent.replace(/[^0-9]/g, '') || '0', 10) : 0;
                    results.push({
                        "날짜": date_text.replace(/\./g, '-').replace(/-$/, ''),
                        "캠페인": cur_cp, "캠페인ID": cur_cp_id, "광고그룹": cur_g, "광고그룹ID": cur_g_id, "광고소재": cur_c, "광고소재ID": cur_c_id,
                        "총비용": clean(cells[idx_cost]), "총전환수": clean(cells[idx_conv]), "총전환매출액": clean(cells[idx_conv_sales])
                    });
                }
            });
            return { states: { cur_c, cur_c_id, cur_g, cur_g_id, cur_cp, cur_cp_id }, items: results };
        }
        """

        all_data = []
        states = {
            "cur_c": "",
            "cur_c_id": "",
            "cur_g": "",
            "cur_g_id": "",
            "cur_cp": "",
            "cur_cp_id": "",
        }

        # [테크닉 3] 가상 돔 청크를 놓치지 않는 50px 간격의 초정밀 슬라이딩 그물망 수집
        print("📥 [Step 6] 0.5cm 미세 간격 런타임 슬라이딩 스캔 시작 (약 1분 소요)...")
        for _ in range(120):
            res = await page.evaluate(JS_EXTRACT_CODE, states)
            states = res["states"]
            all_data.extend(res["items"])
            await page.mouse.wheel(0, 50)  # 50px 단위 마이크로 이동
            await page.wait_for_timeout(100)

        await browser.close()

        # =========================================================================
        # 3. 데이터 후처리 및 정제 (Deduplication & Reindexing)
        # =========================================================================
        print("🧹 [Step 7] 수집 원본 데이터 다차원 중복 검증 및 포맷 최종 정렬...")
        df = pd.DataFrame(all_data)
        if not df.empty:
            # 중복 연산으로 인한 지표 왜곡 방지 (일자별/소재고유ID별 최신 적재 데이터 기준 윈도잉)
            df = df.drop_duplicates(
                subset=["날짜", "광고소재ID"], keep="last"
            ).reset_index(drop=True)

            # 전사 통합 데이터마트 규격에 맞춰 칼럼 스키마 인덱싱 변경
            cols = [
                "날짜",
                "캠페인",
                "광고그룹",
                "광고소재",
                "총비용",
                "총전환수",
                "총전환매출액",
                "캠페인ID",
                "광고그룹ID",
                "광고소재ID",
            ]
            df = df[cols]

        return df


# =========================================================================
# 4. 데이터 파이프라인 가동 및 결과 시각화
# =========================================================================
df_naver = await get_gfa_daily_report()

if not df_naver.empty:
    print(
        f"\n✅ [최종성공] GFA 실적 적재 완료! 데이터 규모: {len(df_naver)} rows"
    )
    display(df_naver)
else:
    print("\n⚠️ [확인 필요] 파이프라인 수집 단계에서 적재된 데이터가 존재하지 않습니다.")

In [ ]:
# =========================================================================
# 5. 데이터 무결성 검증 및 최종 중복 제거 (Data Integrity & Deduplication)
# =========================================================================

# [기본 처리] 데이터셋 내 모든 차원과 지표가 100% 일치하는 완전 중복 행 제거
df_naver = df_naver.drop_duplicates().reset_index(drop=True)

# [고급 테크닉] 네트워크 지연 또는 가상 DOM 중복 렌더링으로 인해
# 수치형 지표(비용/전환)가 미세하게 파편화되어 유입된 경우를 방지하기 위한 비즈니스 키 기반 디듀플리케이션
# (일자별 + 소재 고유 식별자 조합을 Unique Key로 정의하여 정합성 보장)
# df_naver = df_naver.drop_duplicates(subset=['날짜', '광고소재ID'], keep='last').reset_index(drop=True)

print(f"✅ [Deduplication 완료] 데이터 정성 검증 통과! 최종 유니크 행 수: {len(df_naver)} rows")
display(df_naver)

In [ ]:
# 기존 파이프라인 호환을 위한 컬럼 순서 재배치
# (기본 데이터 7개 먼저 -> ID 데이터 3개 나중)
pipeline_order = [
    '날짜', '캠페인', '광고그룹', '광고소재', '총비용', '총전환수', '총전환매출액',
    '캠페인ID', '광고그룹ID', '광고소재ID'
]

# df_naver에 새로운 컬럼 순서 적용
df_naver = df_naver[pipeline_order]

df_naver.head()

In [ ]:
df_naver['총비용'] = df_naver['총비용'] / 1.1

In [ ]:
import json
from google.cloud import storage
import gspread
import pandas as pd

# =========================================================================
# 6. GCS 기반 보안 인증 및 구글 스프레드시트 API 연동 (Sheets API Ingestion)
# =========================================================================
BUCKET_NAME = "YOUR_SECURE_BUCKET"
KEY_FILE_IN_BUCKET = "YOUR_SERVICE_ACCOUNT_KEY.json"

# GCS 버킷으로부터 보안 인증서(Service Account Key) 실시간 스트리밍 로드
storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(KEY_FILE_IN_BUCKET)
key_file_dict = json.loads(blob.download_as_string())

# OAuth 2.0 스코프 정의 및 gspread 클라이언트 핸들러 초기화
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]
gc = gspread.service_account_from_dict(key_file_dict, scopes=SCOPES)

# 고유 Key 스트링을 활용하여 대상 스프레드시트 및 워크시트 인스턴스 오픈
SHEET_ID = "YOUR_SPREADSHEET_ID"
worksheet = gc.open_by_key(SHEET_ID).worksheet("gfa raw")


# =========================================================================
# 7. 기존 적재 데이터 이력 로드 (Historical Data Sync)
# =========================================================================
print("📥 [Sheets API] 대상 워크시트로부터 기존 GFA 실적 이력을 로드하는 중...")

# API 레코드 추출 및 판다스 데이터프레임 구조화
existing_data = worksheet.get_all_records()
df_existing = pd.DataFrame(existing_data)

if not df_existing.empty:
    print(
        f"✅ [동기화 성공] 기존 데이터 동기화 완료! 현재 적재 규모: {len(df_existing)} rows"
    )
else:
    print(
        "ℹ️ [알림] 워크시트가 비어있거나 기존 데이터가 존재하지 않습니다. (신규 적재 대상)"
    )

In [ ]:
# =========================================================================
# 8. Idempotent Upsert 로직 구현 (Incremental Update via Date Windowing)
# =========================================================================

# 신규 수집된 네이버 GFA 실적 데이터(df_naver)의 고유 날짜 윈도우 추출
target_dates = df_naver["날짜"].unique().tolist()

if not df_existing.empty:
    # 지표 왜곡 방지 및 멱등성 보장을 위해, 기존 데이터 중 신규 수집 대상일과 겹치는 날짜 제거 (Overwrite Window)
    df_filtered = df_existing[
        ~df_existing["날짜"].astype(str).isin(target_dates)
    ]
    # 필터링된 과거 이력과 신규 정제 데이터 병합
    df_updated = pd.concat([df_filtered, df_naver], ignore_index=True)
else:
    df_updated = df_naver

# =========================================================================
# 9. 데이터 스키마 안정화 및 확정 정렬 (Schema Stabilization & Sorting)
# =========================================================================

# 타깃 적재소(Google Sheets/BigQuery)의 암묵적 문자열 변환 방지를 위한 명시적 수치형 캐스팅
df_updated["총비용"] = (
    pd.to_numeric(df_updated["총비용"], errors="coerce").fillna(0).astype(int)
)
df_updated["총전환수"] = (
    pd.to_numeric(df_updated["총전환수"], errors="coerce").fillna(0).astype(int)
)
df_updated["총전환매출액"] = (
    pd.to_numeric(df_updated["총전환매출액"], errors="coerce")
    .fillna(0)
    .astype(int)
)

# 다운스트림 파이프라인 및 시각화 인덱싱 가시성을 위한 결정론적 정렬(Deterministic Sorting)
df_updated = df_updated.sort_values(
    by=["날짜", "캠페인"], ascending=[True, True]
).reset_index(drop=True)

print(
    f"✨ [Upsert 완료] 신규 윈도우 반영 및 데이터 마트 병합 성공! 최종 마트 규모: {len(df_updated)} rows"
)

In [ ]:
# --- 5. 구글 시트에 전체 다시 쏘기 (Overwrite) ---
print("시트 업데이트 중...")

# NaN 값 처리 (숫자 컬럼 외의 빈 값은 빈 문자열로)
df_for_upload = df_updated.fillna('')

# 데이터프레임을 리스트 형태로 변환 (헤더 포함)
data_to_upload = [df_for_upload.columns.values.tolist()] + df_for_upload.values.tolist()

# 1. 기존 시트 내용 완전히 비우기
worksheet.clear()

# 2. A1 셀부터 업데이트
# (gspread의 update는 파이썬의 int/float 타입을 구글 시트의 숫자 형식으로 자동 인식합니다)
worksheet.update(data_to_upload, 'A1')

print(f"✅ 업데이트 완료: {target_dates} 기간의 데이터가 갱신되었습니다.")